# Novelty classification

Classify each AI-generated structure by direct training-set match, relaxed substitution match, or no match.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tabulate import tabulate

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_utils import find_repo_root  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
for import_path in (ROOT, NOTEBOOKS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import (  # noqa: E402
    CATEGORY_LABELS,
    CATEGORY_ORDER,
    CRYSTAL_SYSTEM_ORDER,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    load_generated_count,
    missing_required_paths,
    required_paths,
)
from plot_style import (  # noqa: E402
    CATEGORY_COLORS,
    GRAY,
    WHITE,
    apply_plot_style,
)

from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402

ROOT, INPUT_DIR, RESULTS_DIR

In [ ]:
CLASSIFICATION_PATH_KEYS = (
    "generated_structures",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
    "wyckoff_repr",
)


def discover_models() -> list[str]:
    if not RESULTS_DIR.exists():
        return []
    return sorted(
        path.name
        for path in RESULTS_DIR.iterdir()
        if path.is_dir() and path.name != "train" and not path.name.startswith(".")
    )


def model_required_paths(model: str) -> dict[str, Path]:
    return required_paths(model, INPUT_DIR, RESULTS_DIR, CLASSIFICATION_PATH_KEYS)

In [ ]:
models = discover_models()
missing_files = pd.DataFrame(
    record
    for model in models
    for record in missing_required_paths(model_required_paths(model), model=model)
)
complete_models = [
    model
    for model in models
    if missing_files.empty or model not in set(missing_files["model"])
]

classifications = (
    pd.concat(
        [
            classify_model(
                model,
                model_required_paths(model),
                include_category_label=True,
                include_crystal_system=True,
            )
            for model in complete_models
        ],
        ignore_index=True,
    )
    if complete_models
    else pd.DataFrame(
        columns=[
            "model",
            "gen_idx",
            "category",
            "category_label",
            "crystal_system",
            "is_direct_sm_match",
            "has_relaxed_sm_anon_match",
            "has_relaxed_wyckoff_match",
            "ehull_relaxed",
            "is_metastable",
            "is_smact_valid",
            "is_metastable_smact_valid",
        ]
    )
)

if not classifications.empty:
    classifications["category"] = pd.Categorical(
        classifications["category"], categories=CATEGORY_ORDER, ordered=True
    )

models, complete_models

In [ ]:
def build_summary(classifications: pd.DataFrame) -> pd.DataFrame:
    if classifications.empty:
        return pd.DataFrame(columns=["model", "category", "count", "percent"])

    counts = (
        classifications.groupby(["model", "category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [complete_models, CATEGORY_ORDER], names=["model", "category"]
    )
    summary = counts.reindex(full_index, fill_value=0).reset_index()
    totals = summary.groupby("model")["count"].transform("sum")
    summary["percent"] = np.where(totals > 0, 100 * summary["count"] / totals, 0.0)
    summary["category_label"] = summary["category"].map(CATEGORY_LABELS)
    return summary


if not classifications.empty:
    expected_counts = {
        model: load_generated_count(model, INPUT_DIR) for model in complete_models
    }
    observed_counts = classifications.groupby("model").size().to_dict()
    assert observed_counts == expected_counts, (observed_counts, expected_counts)

    direct_non_category_1 = classifications[
        classifications["is_direct_sm_match"] & (classifications["category"] != "1")
    ]
    assert direct_non_category_1.empty

    category_1_not_direct = classifications[
        (classifications["category"] == "1") & ~classifications["is_direct_sm_match"]
    ]
    assert category_1_not_direct.empty

    metastable_smact_valid_mismatch = classifications[
        classifications["is_metastable_smact_valid"]
        != (classifications["is_metastable"] & classifications["is_smact_valid"])
    ]
    assert metastable_smact_valid_mismatch.empty

## Integrated summaries

Novelty category statistics for all generated samples, metastable samples (`Ehull <= 0.1 eV/atom`), and metastable SMACT-valid samples.

In [ ]:
SUBSET_MASKS = {
    "all": pd.Series(True, index=classifications.index),
    "metastable": classifications["is_metastable"],
    "metastable_smact_valid": classifications["is_metastable_smact_valid"],
}


def build_subset_summary(classifications: pd.DataFrame) -> pd.DataFrame:
    if classifications.empty:
        return pd.DataFrame(
            columns=[
                "subset",
                "model",
                "category",
                "count",
                "percent",
                "category_label",
            ]
        )

    summaries = []
    for subset, mask in SUBSET_MASKS.items():
        subset_summary = build_summary(classifications[mask]).copy()
        subset_summary.insert(0, "subset", subset)
        summaries.append(subset_summary)
    return pd.concat(summaries, ignore_index=True)


def format_count_percent(count: int, percent: float) -> str:
    return f"{count:,} ({percent:.1f}%)"


def build_tabulated_summary(summary_by_subset: pd.DataFrame) -> str:
    if summary_by_subset.empty:
        return "No classifications available."

    display_summary = summary_by_subset.copy()
    display_summary["value"] = [
        format_count_percent(count, percent)
        for count, percent in zip(
            display_summary["count"], display_summary["percent"], strict=True
        )
    ]
    table = display_summary.pivot(
        index=["model", "subset"], columns="category", values="value"
    ).reindex(columns=CATEGORY_ORDER)
    table = table.rename(columns=CATEGORY_LABELS)
    table = table.reset_index().rename_axis(columns=None)
    table = table.fillna(format_count_percent(0, 0.0))
    return tabulate(table, headers="keys", tablefmt="github", showindex=False)


summary_by_subset = build_subset_summary(classifications)

if not classifications.empty:
    subset_sizes = {
        subset: int(mask.groupby(classifications["model"]).sum().sum())
        for subset, mask in SUBSET_MASKS.items()
    }
    observed_subset_counts = (
        summary_by_subset.groupby("subset")["count"].sum().to_dict()
    )
    assert observed_subset_counts == subset_sizes, (
        observed_subset_counts,
        subset_sizes,
    )

display(Markdown(build_tabulated_summary(summary_by_subset)))

## Category counts by crystal system

For each model, stacked bars show the novelty-category counts within each crystal system.

In [ ]:
apply_plot_style()

CRYSTAL_SYSTEM_SUBSET_LABELS = {
    "all": "All structures",
    "metastable": "Metastable structures",
    "metastable_smact_valid": "Metastable and SMACT-valid structures",
}


def build_crystal_system_category_counts(classifications: pd.DataFrame) -> pd.DataFrame:
    if classifications.empty:
        return pd.DataFrame(columns=["model", "crystal_system", "category", "count"])

    counts = (
        classifications.groupby(["model", "crystal_system", "category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [complete_models, CRYSTAL_SYSTEM_ORDER, CATEGORY_ORDER],
        names=["model", "crystal_system", "category"],
    )
    return counts.reindex(full_index, fill_value=0).reset_index()


def build_tabulated_crystal_system_counts(counts_by_system: pd.DataFrame) -> str:
    if counts_by_system.empty:
        return "No classifications available."

    full_index = pd.MultiIndex.from_product(
        [complete_models, CRYSTAL_SYSTEM_ORDER],
        names=["model", "crystal_system"],
    )
    counts_table = (
        counts_by_system.pivot(
            index=["model", "crystal_system"], columns="category", values="count"
        )
        .reindex(index=full_index, columns=CATEGORY_ORDER)
        .fillna(0)
        .astype(int)
    )
    totals = counts_table.sum(axis=1)
    counts_table = counts_table[totals > 0]
    totals = totals[totals > 0]

    table = counts_table.copy()
    for category in CATEGORY_ORDER:
        table[category] = [
            f"{count:,} ({100 * count / total:.1f}%)"
            for count, total in zip(counts_table[category], totals, strict=True)
        ]
    table = (
        table.rename(columns=CATEGORY_LABELS).reset_index().rename_axis(columns=None)
    )
    return tabulate(table, headers="keys", tablefmt="github", showindex=False)


def plot_category_counts_by_crystal_system(counts_by_system: pd.DataFrame) -> None:
    if counts_by_system.empty:
        display(Markdown("No classifications available."))
        return

    hatch_by_category = {"2-1": "///"}
    models_with_data = [
        model
        for model in complete_models
        if counts_by_system.loc[counts_by_system["model"] == model, "count"].sum() > 0
    ]
    if not models_with_data:
        display(Markdown("No classifications available."))
        return

    fig, axes = plt.subplots(
        len(models_with_data),
        1,
        figsize=(8.0, 2.6 * len(models_with_data)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    for ax, model in zip(axes.ravel(), models_with_data, strict=True):
        model_counts = counts_by_system[counts_by_system["model"] == model]
        values = (
            model_counts.pivot(
                index="crystal_system", columns="category", values="count"
            )
            .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=CATEGORY_ORDER)
            .fillna(0.0)
        )
        counts = (
            model_counts.groupby("crystal_system", observed=False)["count"]
            .sum()
            .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
        )
        systems_with_data = counts[counts > 0].index.tolist()
        values = values.loc[systems_with_data]

        bottom = np.zeros(len(values), dtype=int)
        x = np.arange(len(values))
        for category in CATEGORY_ORDER:
            heights = values[category].to_numpy()
            edgecolor = CATEGORY_COLORS["2-3"] if category == "2-1" else WHITE
            linewidth = 0.8 if category == "2-1" else 0.5
            ax.bar(
                x,
                heights,
                bottom=bottom,
                color=CATEGORY_COLORS[category],
                edgecolor=edgecolor,
                linewidth=linewidth,
                hatch=hatch_by_category.get(category),
                label=CATEGORY_LABELS[category],
            )
            bottom += heights

        ax.set_title(model)
        ax.set_ylabel("structures")
        ax.set_xticks(x, systems_with_data, rotation=30, ha="right")
        ax.grid(axis="y", color=GRAY, alpha=0.35, linewidth=0.6)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    legend_handles = [
        mpatches.Patch(
            facecolor=CATEGORY_COLORS[category],
            edgecolor=CATEGORY_COLORS["2-3"] if category == "2-1" else WHITE,
            hatch=hatch_by_category.get(category),
            label=CATEGORY_LABELS[category],
        )
        for category in CATEGORY_ORDER
    ]
    axes.ravel()[0].legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=False,
    )
    fig.tight_layout()
    plt.show()


crystal_system_category_counts_by_subset = {}
for subset, label in CRYSTAL_SYSTEM_SUBSET_LABELS.items():
    subset_counts = build_crystal_system_category_counts(
        classifications[SUBSET_MASKS[subset]]
    )
    crystal_system_category_counts_by_subset[subset] = subset_counts
    display(Markdown(f"### {label}"))
    display(Markdown(build_tabulated_crystal_system_counts(subset_counts)))
    plot_category_counts_by_crystal_system(subset_counts)

## Missing files

Incomplete model directories are skipped. If this table is empty, all discovered model directories were complete.

In [ ]:
missing_files

## Per-structure classifications

In [ ]:
classifications